In [4]:
import urllib.request
import zipfile

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
zip_path = "drugsCom_raw.zip"

urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall() # Extracts into the current directory



In [7]:
from datasets import load_dataset

data_files = {"train": "drugsComTrain_raw.tsv" , "test": "drugsComTest_raw.tsv"}

drug_dataset = load_dataset("csv",data_files = data_files , delimiter = "\t")


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [10]:
drug_sample = drug_dataset["train"].shuffle(seed = 234).select(range(100))

In [13]:
drug_sample[:1]

{'Unnamed: 0': [125858],
 'drugName': ['Viibryd'],
 'condition': ['Depression'],
 'review': ['"This started out as a great experience. I was on Lexapro but due to the decreased sexual desire wanted to try something else.\r\r\nThe first three months felt great, no improvement on the sexual aspect however.  Then the side effects started and it was terrible. Started with constipation, which I was able to put up with. The legs hurting and cramping was another story. As long as I was active I had no problem. Go on a trip and it was hell. My legs would hurt like crazy. Then at night in bed if I turned over I got leg cramps in my thighs.\r\r\n\r\r\nOH and the DREAMS! So real, vivid and weird.\r\r\n\r\r\nExpensive!"'],
 'rating': [4.0],
 'date': ['June 7, 2016'],
 'usefulCount': [20]}

This gives us an accurate view on our dataset 

In [15]:
for split in drug_dataset.keys():
    assert len(drug_dataset[split]) == len((drug_dataset[split]).unique("Unnamed: 0"))

This means that all the patient names in our dataset are : "Unnamed: 0"

In [18]:
drug_dataset = drug_dataset.rename_column(
    original_column_name = "Unnamed: 0",
    new_column_name = "patient_id",
)

ValueError: Original column name Unnamed: 0 not in the dataset. Current columns in the dataset: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount']

Map:   0%|          | 0/161297 [00:00<?, ? examples/s]

AttributeError: 'NoneType' object has no attribute 'lower'

In [28]:
drug_dataset = drug_dataset.filter(lambda x : x['condition'] is not None)

def lowercase_function(example): 
    return {"condition" : example['condition'].lower()}

Filter:   0%|          | 0/161297 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53766 [00:00<?, ? examples/s]

In [30]:
drug_dataset = drug_dataset.map(lowercase_function)

In [37]:
drug_dataset["train"]["condition"][:7]

['left ventricular dysfunction',
 'adhd',
 'birth control',
 'birth control',
 'opiate dependence',
 'benign prostatic hyperplasia',
 'emergency contraception']

In [38]:
def compute_review_length(example): 
    return {"review_length": len(example["review"].split())}

In [39]:
drug_dataset = drug_dataset.map(compute_review_length)

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

In [40]:
drug_dataset = drug_dataset.filter(lambda x : x["review_length"] < 30)

Filter:   0%|          | 0/160398 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53471 [00:00<?, ? examples/s]

In [45]:
drug_dataset["train"]["review_length"][:3]

[17, 27, 27]

In [48]:
%pip install html

Defaulting to user installation because normal site-packages is not writeable
  Using cached html-1.16.tar.gz (7.6 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  python setup.py egg_info did not run successfully.
  exit code: 1
  
  [17 lines of output]
  Traceback (most recent call last):
    File "<string>", line 2, in <module>
      exec(compile('''
      ~~~~^^^^^^^^^^^^
      # This is <pip-setuptools-caller> -- a caller that pip uses to run setup.py
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
      ...<32 lines>...
      exec(compile(setup_py_code, filename, "exec"))
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
      ''' % ('C:\\Users\\Lenovo\\AppData\\Local\\Temp\\pip-install-cb82eu2y\\html_f2f6462cb2b34b028aba6540ad8a2058\\setup.py',), "<pip-setuptools-caller>", "exec"))
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "<pip-setuptools-caller>", line 35, in <module>
    File "C:\Users\Lenovo\AppData\Local\Temp\pip-install-cb82

In [52]:
import html 

drug_dataset = drug_dataset.map(lambda x : {"review" : html.unescape(x["review"])})  

Map:   0%|          | 0/20869 [00:00<?, ? examples/s]

Map:   0%|          | 0/7037 [00:00<?, ? examples/s]

In [53]:
import pandas

drug_dataset.set_format("pandas")

In [56]:
train_df = drug_dataset["train"][:]

In [73]:
frequencies = (
    train_df["condition"].
        value_counts().
        to_frame().
        reset_index().
        rename(columns = {"index":"condition","count":"frequency"})
    
)

frequencies.sort_values(by = "frequency",ascending = False).head(10)

,condition,frequency
0,pain,1357
1,birth control,1048
2,depression,1012
3,anxiety,871
4,insomnia,665
5,bipolar disorde,600
6,high blood pressure,507
7,adhd,395
8,"diabetes, type 2",368
9,acne,363


In [76]:
from datasets import Dataset
freq_dataset = Dataset.from_pandas(frequencies)

In [78]:
freq_dataset

Dataset({
    features: ['condition', 'frequency'],
    num_rows: 634
})